# Step 00: Multi-Source COCO Dataset Ingestion & Image Caching
### Ingest, Cache & Verify Annotation JSONs and Images into `coco_files/`
Directly extracted from `multi_class_train_rfdetr.ipynb` (Cells 1, 2, 3, 4, 5, 6, 7, 8).

**Strict Scope**:
- **Dataset & Download Only**: No model architecture, weights loading, or training code.
- **Zero Hardcoded Credentials**: Reads Azure SAS / Connection Strings via environment variables or `.env`.
- **Fast Skip-Cache Check**: Validates local file existence before downloading, avoiding redundant network transfers.
- **Shared Image Pool**: Stores images in `coco_files/images/` with symlink support.
- **COCO JSON Format Verification**: Explicitly verifies JSON syntax and standard schema (`images`, `annotations`, `categories`).


In [ ]:
# STEP 0 - Environment Dependencies (Download & Data Ingestion Only)
%pip install -q python-dotenv requests tqdm azure-storage-blob tabulate pillow


In [ ]:
# CELL 1 - Imports & Logger Initialization (Download & Ingestion Tools Only)
import os, sys, json, time, logging, random, shutil, warnings
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
import urllib.parse
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from dotenv import load_dotenv
from tqdm.auto import tqdm

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

logger.info("=" * 65)
logger.info("[CELL 1] Ingestion Logger Ready (Download & Ingestion Scope Only)")
logger.info("=" * 65)


In [ ]:
# CELL 2 - Ingestion & Path Configuration (From multi_class_train_rfdetr.ipynb)
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
PIPELINE_NAME = "multi_class_train_rfdetr"
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME

INPUT_JSON_DIR = REPO_ROOT / "coco_files"
IMAGES_DIR = PIPELINE_DIR / "images"
RAW_IMAGES_DIR = IMAGES_DIR

SAMPLE_SIZE = None  # None for full data ingestion, or integer for sample test
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"
RANDOM_SEED = 42

# Field names in COCO annotations
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

# Azure Blob Storage Configuration
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"
DOWNLOAD_WORKERS = 16

logger.info(f"Target COCO Files Directory: {INPUT_JSON_DIR.resolve()}")
logger.info(f"Target Images Directory:     {IMAGES_DIR.resolve()}")
logger.info(f"Download Worker Threads:     {DOWNLOAD_WORKERS}")


In [ ]:
# CELL 3 - Dynamic Folder Creation & Azure Client Initialization
os.makedirs(INPUT_JSON_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

load_dotenv()
conn_str = os.getenv(AZURE_CONNECTION_STRING_ENV)
try:
    from azure.storage.blob import BlobServiceClient
    blob_service_client = BlobServiceClient.from_connection_string(conn_str) if conn_str else None
except Exception as e:
    blob_service_client = None

if blob_service_client:
    logger.info("Azure Blob Storage client authenticated successfully.")
else:
    logger.info("Azure Blob client not connected. Falling back to direct HTTP/HTTPS URL downloading.")


In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")

In [ ]:
# CELL 6 - Group Annotations & Apply Stratified Sampling Mode
logger.info("=" * 70)
logger.info(f"[CELL 6] Applying mode selection: {MODE_TAG.upper()}")
logger.info("=" * 70)

all_image_urls = sorted(list(image_annotations.keys()))
random.seed(RANDOM_SEED)

image_primary_cat: Dict[str, str] = {}
for u in all_image_urls:
    anns = image_annotations[u]
    if anns:
        cats = [a["category_name"] for a in anns]
        image_primary_cat[u] = max(set(cats), key=cats.count)
    else:
        image_primary_cat[u] = AUTO_CATEGORIES[0]

if SAMPLE_SIZE is not None and len(all_image_urls) > SAMPLE_SIZE:
    selected_image_urls = []
    cat_to_images: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in all_image_urls:
        cat_to_images[image_primary_cat[u]].append(u)
    
    total_imgs = len(all_image_urls)
    for cat in AUTO_CATEGORIES:
        imgs = cat_to_images[cat]
        target_count = max(1, int(round((len(imgs) / total_imgs) * SAMPLE_SIZE)))
        selected = random.sample(imgs, min(len(imgs), target_count))
        selected_image_urls.extend(selected)
    
    if len(selected_image_urls) > SAMPLE_SIZE:
        selected_image_urls = random.sample(selected_image_urls, SAMPLE_SIZE)
    logger.info(f"   Sample Mode Active: Stratified selection of {len(selected_image_urls)} images.")
else:
    selected_image_urls = all_image_urls
    logger.info(f"   Full Data Mode: Using all {len(selected_image_urls)} images.")

image_urls = sorted(list(set(selected_image_urls)))
filtered_annotations = {u: image_annotations[u] for u in image_urls}
total_selected_boxes = sum(len(v) for v in filtered_annotations.values())

logger.info(f"   - Total images in pipeline:         {len(image_urls)}")
logger.info(f"   - Total bounding boxes in pipeline: {total_selected_boxes}")

In [ ]:
# CELL 7 - Azure URL Helpers & Download Function
logger.info("=" * 70)
logger.info("[CELL 7] Defining Azure Blob download and caching functions...")
logger.info("=" * 70)

IMAGES_DIR = PIPELINE_DIR / "images"
os.makedirs(IMAGES_DIR, exist_ok=True)

def parse_azure_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    parsed = urlparse(url)
    parts = parsed.path.strip("/").split("/", 1)
    if len(parts) == 2:
        return parts[0], parts[1]
    return None, None

def download_image(url: str, dest_dir: Path) -> Tuple[str, Optional[Path], Optional[str]]:
    filename = Path(urlparse(url).path).name or os.path.basename(url)
    dest_path = Path(dest_dir) / filename
    
    # Fast check: skip download if image already exists and is non-empty
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return url, dest_path, None

    dest_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = dest_path.with_suffix(dest_path.suffix + ".part")

    container_name, blob_name = parse_azure_url(url)
    if blob_service_client and container_name and blob_name:
        try:
            blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
            with open(temp_path, "wb") as f:
                f.write(blob_client.download_blob().readall())
            os.replace(temp_path, dest_path)
            return url, dest_path, None
        except Exception as e:
            if temp_path.exists():
                temp_path.unlink()

    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            with open(temp_path, "wb") as f:
                f.write(resp.content)
            os.replace(temp_path, dest_path)
            return url, dest_path, None
        else:
            if temp_path.exists():
                temp_path.unlink()
            return url, None, f"HTTP {resp.status_code}"
    except Exception as e:
        if temp_path.exists():
            temp_path.unlink()
        return url, None, str(e)

logger.info("Image download functions ready (with cache skipping).")


In [ ]:
# CELL 8 - Parallel Azure Image Downloads (With Local Caching & Skip)
logger.info("=" * 70)
logger.info(f"[CELL 8] Checking local cache for {len(image_urls)} images...")
logger.info("=" * 70)

dest_dir = Path(IMAGES_DIR)
dest_dir.mkdir(parents=True, exist_ok=True)

download_results: Dict[str, Dict[str, Any]] = {}
to_download: List[str] = []

# Pre-check: skip images already present locally with non-zero size
for url in image_urls:
    fname = Path(urlparse(url).path).name or os.path.basename(url)
    local_file = dest_dir / fname
    if local_file.exists() and local_file.stat().st_size > 0:
        download_results[url] = {"local_path": local_file, "error": None}
    else:
        to_download.append(url)

already_cached = len(download_results)
logger.info(f"   - Already cached locally: {already_cached} / {len(image_urls)} images (skipped download).")
logger.info(f"   - Remaining to download:  {len(to_download)} images.")

if to_download:
    logger.info(f"   - Downloading {len(to_download)} images ({DOWNLOAD_WORKERS} threads)...")
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
        futures = {executor.submit(download_image, url, dest_dir): url for url in to_download}
        done_count = 0
        for future in as_completed(futures):
            url, path, err = future.result()
            download_results[url] = {"local_path": path, "error": err}
            done_count += 1
            if done_count % 200 == 0 or done_count == len(to_download):
                logger.info(f"   - Download Progress: {done_count}/{len(to_download)} processed ({done_count/len(to_download)*100:.1f}%)...")
else:
    logger.info("   - All images already present locally. Download skipped completely!")

success_downloads = [u for u, res in download_results.items() if res["error"] is None]
failed_downloads = [u for u, res in download_results.items() if res["error"] is not None]

logger.info("Download Summary:")
logger.info(f"   - Successfully ready: {len(success_downloads)} images")
logger.info(f"   - Failed:             {len(failed_downloads)} images")


In [ ]:
# CELL 9 - Verify COCO Files Are Valid JSON & Schema Compliant
def verify_coco_files_are_json(directory: Path):
    """Checks that all files intended as annotations are valid JSON files conforming to COCO format."""
    all_files = [p for p in directory.glob("*") if p.is_file() and not p.name.startswith(".")]
    json_files = [p for p in all_files if p.suffix.lower() == ".json"]
    non_json = [p for p in all_files if p.suffix.lower() not in [".json", ".md"]]
    
    logger.info("=" * 70)
    logger.info(f"[VALIDATION] Verifying COCO JSON files in: {directory.resolve()}")
    if non_json:
        logger.warning(f"Non-JSON files detected in annotations folder: {[p.name for p in non_json]}")
        
    if not json_files:
        logger.info("No .json files found in coco_files/ yet. Place or download annotations before proceeding.")
        return []
        
    valid_coco = []
    for jf in json_files:
        try:
            with open(jf, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict):
                req = ["images", "annotations", "categories"]
                missing = [k for k in req if k not in data]
                if not missing:
                    logger.info(f"✓ Valid COCO JSON: '{jf.name}' ({len(data['images'])} images, {len(data['annotations'])} annotations, {len(data['categories'])} categories)")
                    valid_coco.append(jf)
                else:
                    logger.warning(f"File '{jf.name}' is valid JSON, but missing standard COCO keys: {missing}")
            elif isinstance(data, list):
                logger.info(f"✓ Valid JSON List: '{jf.name}' ({len(data)} records)")
                valid_coco.append(jf)
            else:
                logger.warning(f"File '{jf.name}' is JSON, but root entity is {type(data).__name__} instead of dict/list.")
        except json.JSONDecodeError as err:
            logger.error(f"✗ File '{jf.name}' is NOT valid JSON: {err}")
    logger.info("=" * 70)
    return valid_coco

verified_coco_files = verify_coco_files_are_json(INPUT_JSON_DIR)


In [ ]:
# CELL 10 - Summary Audit & Next Step Transition
cached_images = list(Path(IMAGES_DIR).glob("*"))
logger.info("=" * 70)
logger.info(" STEP 00 INGESTION COMPLETED")
logger.info(f"   - Verified COCO JSON files:    {len(verified_coco_files)}")
logger.info(f"   - Total images cached locally: {len(cached_images):,}")
logger.info(f"   - Discovered categories:       {AUTO_CATEGORIES}")
logger.info(" Next: Proceed to '01_eda_annotations.ipynb' for exploratory data analysis.")
logger.info("=" * 70)
